In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import warnings
import sys
import re
import html

# ===================== 屏蔽警告 =====================
warnings.filterwarnings('ignore')
sys.dont_write_bytecode = True
if not sys.warnoptions:
    warnings.simplefilter("ignore")

# ===================== 初始化SparkSession =====================
spark = SparkSession.builder \
    .appName("OA_Workflow_Node_Detail_Analysis") \
    .enableHiveSupport() \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .getOrCreate()

# ===================== 表名定义 =====================
TABLE_REQUEST_BASE = "ods.ods_oa_workflow_requestbase_i_1h"
TABLE_CURRENT_OPERATOR = "ods.ods_oa_workflow_currentoperator_i_1h"
TABLE_WORKFLOW_BASE = "ods.ods_oa_workflow_base_f_1d"
TABLE_REQUEST_LOG = "ods.ods_oa_workflow_requestlog_i_1h"
TABLE_HRM_RESOURCE = "ods.ods_oa_hrmresource_f_1d"
TABLE_HRM_DEPARTMENT = "ods.ods_oa_hrmdepartment_f_1d"
TABLE_HRM_JOBTITLES = "ods.ods_oa_hrmjobtitles_f_1d"
TABLE_HRM_SUBCOMPANY = "ods.ods_oa_hrmsubcompany_f_1d"
TABLE_WORKFLOW_NODEBASE = "ods.ods_oa_workflow_nodebase_f_1d"
target_table = "dwd.dwd_oa_workflow_efficiency_nodes_f_1h"

# ===================== 加载所有Hive表 =====================
df_request_base = spark.table(TABLE_REQUEST_BASE).alias("T1")
df_current_operator = spark.table(TABLE_CURRENT_OPERATOR).alias("T2")
df_workflow_base = spark.table(TABLE_WORKFLOW_BASE).alias("T3")
df_request_log = spark.table(TABLE_REQUEST_LOG).alias("T7")
df_hrm_resource = spark.table(TABLE_HRM_RESOURCE).alias("T4")
df_hrm_department = spark.table(TABLE_HRM_DEPARTMENT).alias("T6")
df_hrm_jobtitles = spark.table(TABLE_HRM_JOBTITLES).alias("T5")
df_hrm_subcompany = spark.table(TABLE_HRM_SUBCOMPANY).alias("Company")
df_workflow_nodebase = spark.table(TABLE_WORKFLOW_NODEBASE).alias("T8")

# ===================== 工具函数 =====================
def clean_chinese_text(text):
    if not text:
        return ""
    return re.sub(r"[^\u4e00-\u9fa5]", "", text)

clean_chinese_udf = F.udf(clean_chinese_text, StringType())

def decode_unicode_escapes(text):
    text = re.sub(r'\\u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), text)
    text = re.sub(r'\\U([0-9a-fA-F]{8})', lambda m: chr(int(m.group(1), 16)), text)
    return text

def clean_remark(text):
    if not text:
        return ""
    text = html.unescape(text)
    text = decode_unicode_escapes(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # 仅保留中文、英文、数字、空格、常用标点
    allowed_pattern = r'[^\u4e00-\u9fa5\u0020-\u007F\u3000-\u303f\uff00-\uffef]'
    text = re.sub(allowed_pattern, '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_html_udf = F.udf(clean_remark, StringType())

# ===================== 核心子查询 =====================
# 4.1 流程基础信息（仅保留状态为“办结” 或 预算 的流程）
df_wf_base = df_request_base.alias("T1") \
    .filter( (F.col("T1.status") == "办结") | (F.col("T1.requestname").contains("预算")) ) \
    .filter(F.to_date(F.col("T1.createdate")) > F.to_date(F.lit("2022-01-01")))\
    .select(
        F.col("T1.requestid").alias("requestid"),
        F.col("T1.requestname").alias("process_name"),
        F.col("T1.createdate").alias("process_create_date"),
        F.col("T1.status").alias("process_status"),   # 直接取原始 status
        F.col("T1.requestmark").alias("request_mark")
    )

# 4.2 流程名称
df_wf_name = df_current_operator.alias("T2") \
    .join(df_workflow_base.alias("T3"), F.col("T2.workflowid") == F.col("T3.id"), "left") \
    .select(
        F.col("T2.requestid").alias("requestid"),
        F.col("T2.id").alias("node_id"),
        F.coalesce(clean_chinese_udf(F.col("T3.workflowname")), F.lit("无流程类型")).alias("workflow_name")
    )

# 4.3 节点核心信息
df_node_core = df_current_operator.alias("T2").select(
    F.col("T2.requestid").alias("requestid"),
    F.col("T2.id").alias("node_id"),
    F.col("T2.nodeid").alias("node_code"),
    F.col("T2.workflowid").alias("workflow_id"),
    F.col("T2.userid").alias("approver_user_id"),
    F.col("T2.showorder").alias("node_show_order"),
    F.col("T2.operatedate").alias("operatedate"),
    F.col("T2.operatetime").alias("operatetime"),
    F.coalesce(
        F.when(
            F.col("T2.receivedate").isNotNull() & F.col("T2.receivetime").isNotNull(),
            F.to_timestamp(F.concat(F.col("T2.receivedate"), F.lit(" "), F.col("T2.receivetime")))
        ),
        F.lit(None).cast("timestamp")
    ).alias("node_receive_time"),
    F.coalesce(
        F.when(
            F.col("T2.operatedate").isNotNull() & F.col("T2.operatetime").isNotNull(),
            F.to_timestamp(F.concat(F.col("T2.operatedate"), F.lit(" "), F.col("T2.operatetime")))
        ),
        F.lit(None).cast("timestamp")
    ).alias("node_operate_time"),
    F.when(
        F.col("T2.receivedate").isNotNull() & F.col("T2.receivetime").isNotNull() &
        F.col("T2.operatedate").isNotNull() & F.col("T2.operatetime").isNotNull(),
        F.round(
            (F.unix_timestamp(F.concat(F.col("T2.operatedate"), F.lit(" "), F.col("T2.operatetime"))) - 
             F.unix_timestamp(F.concat(F.col("T2.receivedate"), F.lit(" "), F.col("T2.receivetime")))) / 3600,
            2
        )
    ).cast(DoubleType()).alias("node_cost_hours")
)

# 4.4 节点名称
df_node_name = df_node_core.alias("node") \
    .join(df_workflow_nodebase.alias("T8"), F.col("node.node_code") == F.col("T8.id"), "left") \
    .select(
        F.col("node.requestid").alias("requestid"),
        F.col("node.node_id").alias("node_id"),
        F.coalesce(clean_chinese_udf(F.col("T8.nodename")), F.lit("无节点名称")).alias("node_name")
    )

# 4.5 审批人明细
df_approver_info = df_hrm_resource.alias("T4").select(
    F.col("T4.id").alias("approver_user_id"),
    F.col("T4.lastname").alias("approver_name"),
    F.col("T4.jobactivitydesc").alias("approver_job_level"),
    F.col("T4.departmentid").alias("dept_id"),
    F.col("T4.subcompanyid1").alias("sub_company_id"),
    F.col("T4.jobtitle").alias("job_title_id")
)

df_job_title = df_hrm_jobtitles.alias("T5").select(
    F.col("T5.id").alias("job_title_id"),
    F.col("T5.jobtitlemark").alias("approver_job_title")
)

df_dept_org = df_hrm_department.alias("Dep") \
    .join(df_hrm_department.alias("supDept"), F.col("Dep.supdepid") == F.col("supDept.id"), "left") \
    .join(df_hrm_department.alias("supDept1"), F.col("supDept.supdepid") == F.col("supDept1.id"), "left") \
    .join(df_hrm_department.alias("supDept2"), F.col("supDept1.supdepid") == F.col("supDept2.id"), "left") \
    .join(df_hrm_department.alias("supDept3"), F.col("supDept2.supdepid") == F.col("supDept3.id"), "left") \
    .join(df_hrm_subcompany.alias("Company"), F.col("Dep.subcompanyid1") == F.col("Company.id"), "left") \
    .filter((F.col("Dep.canceled").isNull() | (F.col("Dep.canceled") == 0))) \
    .filter(F.col("Company.id").isNull() | ~F.col("Company.id").isin(11, 20, 21, 22, 18)) \
    .select(
        F.col("Dep.id").alias("dept_id"),
        F.col("Dep.departmentname").alias("dept_name"),
        F.when(
            F.concat(
                F.coalesce(F.col("Company.subcompanyname"), F.lit("")), F.lit("||"),
                F.coalesce(F.concat(F.col("supDept3.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept2.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept1.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.col("Dep.departmentname"), F.lit(""))
            ).endswith(">"),
            F.expr("substring(concat(Company.subcompanyname, '||', supDept3.departmentname, '>', supDept2.departmentname, '>', supDept1.departmentname, '>', supDept.departmentname, '>', Dep.departmentname), 1, length(concat(Company.subcompanyname, '||', supDept3.departmentname, '>', supDept2.departmentname, '>', supDept1.departmentname, '>', supDept.departmentname, '>', Dep.departmentname)) - 1)")
        ).otherwise(
            F.concat(
                F.coalesce(F.col("Company.subcompanyname"), F.lit("")), F.lit("||"),
                F.coalesce(F.concat(F.col("supDept3.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept2.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept1.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.concat(F.col("supDept.departmentname"), F.lit(">")), F.lit("")),
                F.coalesce(F.col("Dep.departmentname"), F.lit(""))
            )
        ).alias("dept_full_path")
    )

df_sub_company = df_hrm_subcompany.alias("Company").select(
    F.col("Company.id").alias("sub_company_id"),
    F.col("Company.subcompanyname").alias("sub_company_name")
)

# 4.6 提单人部门（根据节点名称包含“申请人”判断，并去重）
df_requester_dept = df_node_core.alias("core") \
    .join(df_node_name.alias("name"), on=["requestid", "node_id"], how="inner") \
    .filter(F.col("name.node_name").contains("申请人")) \
    .join(df_approver_info.alias("ai"), on="approver_user_id", how="left") \
    .join(df_dept_org.alias("do"), on="dept_id", how="left") \
    .select(
        F.col("core.requestid").alias("requestid"),
        F.coalesce(F.col("do.dept_full_path"), F.lit("未知")).alias("requester_dept_full_path"),
        F.coalesce(F.col("do.dept_name"), F.lit("未知")).alias("requester_dept_name")
    ) \
    .dropDuplicates(["requestid"])   # 确保每个流程只有一个提单人部门信息

# ===================== 节点备注清洗聚合（已统一关联条件） =====================
# 4.7.1 提取节点操作实例的时间信息及审批人ID（用于与日志关联）
df_node_time = df_node_core.select(
    "requestid",
    "node_code",
    "operatedate",
    "operatetime",
    "node_id",
    "approver_user_id"      # 对应日志的 operator
).distinct()

# 4.7.2 从 request_log 读取 remark 和 operator，关联 node_id
df_request_log_remark = df_request_log.select(
    F.col("requestid").alias("log_requestid"),
    F.col("nodeid").alias("log_node_code"),
    F.col("operatedate").alias("log_operatedate"),
    F.col("operatetime").alias("log_operatetime"),
    F.col("operator").alias("log_operator"),
    F.col("remark").alias("raw_remark")
).filter(F.col("raw_remark").isNotNull())

df_request_log_remark = df_request_log_remark.withColumn(
    "cleaned_remark",
    clean_html_udf(F.col("raw_remark"))
).filter(F.col("cleaned_remark") != "")        # 仅保留清洗后非空的备注（不再过滤“来自客户端”）

# 关联获取 node_id：匹配五个字段
df_remark_with_node = df_request_log_remark.alias("log").join(
    df_node_time.alias("node"),
    (F.col("log.log_requestid") == F.col("node.requestid")) &
    (F.col("log.log_node_code") == F.col("node.node_code")) &
    (F.col("log.log_operatedate") == F.col("node.operatedate")) &
    (F.col("log.log_operatetime") == F.col("node.operatetime")) &
    (F.col("log.log_operator") == F.col("node.approver_user_id")),
    "left"
).select(
    F.col("log.log_requestid").alias("requestid"),
    F.col("node.node_id").alias("node_id"),
    F.col("log.cleaned_remark")
).filter(F.col("node_id").isNotNull())  # 过滤无法匹配的日志

# 按 node_id 聚合 remark
df_remark_agg = df_remark_with_node.groupBy("requestid", "node_id").agg(
    F.concat_ws("； ", F.collect_list("cleaned_remark")).alias("node_remark_info")
)

# ===================== 修改点：is_return 使用统一关联条件 =====================
# 4.8 用于判断是否退回的日志（logtype=3），同样匹配五个字段以关联到具体节点实例
df_return_log_raw = df_request_log.filter(F.col("logtype") == "3") \
    .select(
        F.col("requestid").alias("log_requestid"),
        F.col("nodeid").alias("log_node_code"),
        F.col("operatedate").alias("log_operatedate"),
        F.col("operatetime").alias("log_operatetime"),
        F.col("operator").alias("log_operator")
    ).distinct()

# 关联 df_node_time 获取 node_id（使用相同五个字段条件）
df_return_with_node = df_return_log_raw.alias("log").join(
    df_node_time.alias("node"),
    (F.col("log.log_requestid") == F.col("node.requestid")) &
    (F.col("log.log_node_code") == F.col("node.node_code")) &
    (F.col("log.log_operatedate") == F.col("node.operatedate")) &
    (F.col("log.log_operatetime") == F.col("node.operatetime")) &
    (F.col("log.log_operator") == F.col("node.approver_user_id")),
    "left"
).select(
    F.col("node.requestid").alias("requestid"),
    F.col("node.node_id").alias("node_id"),
    F.lit(1).alias("return_flag")
).filter(F.col("node_id").isNotNull())  # 过滤无法匹配的日志

# 按 node_id 去重（同一个节点实例可能有多条退回日志？保险起见）
df_return_agg = df_return_with_node.groupBy("requestid", "node_id").agg(
    F.max("return_flag").alias("return_flag")
)

# ===================== 主数据关联 =====================
df_node_core = df_node_core.alias("core")

df_main = df_node_core \
    .join(df_wf_base.alias("wf"), on="requestid", how="inner") \
    .join(df_wf_name.alias("wn"), on=["requestid", "node_id"], how="left") \
    .join(df_node_name.alias("nn"), on=["requestid", "node_id"], how="left") \
    .join(df_approver_info.alias("ai"), on="approver_user_id", how="left") \
    .join(df_job_title.alias("jt"), on="job_title_id", how="left") \
    .join(df_dept_org.alias("do"), on="dept_id", how="left") \
    .join(df_sub_company.alias("sc"), on="sub_company_id", how="left") \
    .join(df_requester_dept.alias("rd"), on="requestid", how="left") \
    .join(df_return_agg.alias("rl"), on=["requestid", "node_id"], how="left") \
    .join(df_remark_agg.alias("rm"), on=["requestid", "node_id"], how="left")

df_main = df_main.withColumn(
    "is_return",
    F.when(F.col("rl.return_flag").isNotNull(), F.lit("是")).otherwise(F.lit("否"))
).withColumn(
    "node_remark_info",
    F.coalesce(F.col("rm.node_remark_info"), F.lit("无"))
).drop("rl.return_flag")

# ===================== 工作日耗时计算（保持不变） =====================
# 2022-2026 节假日及调休列表
holiday_list = [
    # 2022
    '2022-01-01', '2022-01-02', '2022-01-03',
    '2022-01-31', '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-04', '2022-02-05', '2022-02-06',
    '2022-04-03', '2022-04-04', '2022-04-05',
    '2022-04-30', '2022-05-01', '2022-05-02', '2022-05-03', '2022-05-04',
    '2022-06-03', '2022-06-04', '2022-06-05',
    '2022-09-10', '2022-09-11', '2022-09-12',
    '2022-10-01', '2022-10-02', '2022-10-03', '2022-10-04', '2022-10-05', '2022-10-06', '2022-10-07',
    # 2023
    '2023-01-01', '2023-01-02',
    '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-01-25', '2023-01-26', '2023-01-27',
    '2023-04-05',
    '2023-04-29', '2023-04-30', '2023-05-01', '2023-05-02', '2023-05-03',
    '2023-06-22', '2023-06-23', '2023-06-24',
    '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-04', '2023-10-05', '2023-10-06',
    # 2024
    '2024-01-01',
    '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15', '2024-02-16',
    '2024-04-04', '2024-04-05', '2024-04-06',
    '2024-05-01', '2024-05-02', '2024-05-03',
    '2024-06-10',
    '2024-09-15', '2024-09-16', '2024-09-17',
    '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04', '2024-10-05', '2024-10-06', '2024-10-07',
    # 2025
    '2025-01-01',
    '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-03',
    '2025-04-04', '2025-04-05', '2025-04-06',
    '2025-05-01', '2025-05-02', '2025-05-03',
    '2025-05-31', '2025-06-01', '2025-06-02',
    '2025-10-01', '2025-10-02', '2025-10-03', '2025-10-04', '2025-10-05', '2025-10-06', '2025-10-07',
    # 2026 (预测)
    '2026-01-01',
    '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18', '2026-02-19', '2026-02-20', '2026-02-21',
    '2026-04-04', '2026-04-05', '2026-04-06',
    '2026-05-01', '2026-05-02', '2026-05-03',
    '2026-06-20', '2026-06-21', '2026-06-22',
    '2026-09-27', '2026-09-28', '2026-09-29',
    '2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07',
]

work_weekend_list = [
    # 2022
    '2022-01-29', '2022-01-30',
    '2022-04-02',
    '2022-04-24', '2022-05-07',
    '2022-10-08', '2022-10-09',
    # 2023
    '2023-01-28', '2023-01-29',
    '2023-04-23',
    '2023-05-06',
    '2023-06-25',
    '2023-10-07', '2023-10-08',
    # 2024
    '2024-02-04', '2024-02-18',
    '2024-04-07',
    '2024-04-28', '2024-05-11',
    '2024-09-14',
    '2024-09-29', '2024-10-12',
    # 2025
    '2025-01-26',
    '2025-02-08',
    '2025-04-27',
    '2025-05-06',
    '2025-06-28', '2025-10-11',
    # 2026 (预测)
    '2026-02-14', '2026-02-22',
    '2026-04-03',
    '2026-04-26', '2026-05-09',
    '2026-06-13',
    '2026-09-20', '2026-10-10',
]

# 转换为DataFrame
holiday_df = spark.createDataFrame([(d,) for d in holiday_list], ["holiday_date"])
work_weekend_df = spark.createDataFrame([(d,) for d in work_weekend_list], ["work_weekend_date"])

# 计算工作日耗时
df_time_calc = df_main.filter(
    F.col("node_receive_time").isNotNull() & F.col("node_operate_time").isNotNull()
).select(
    "requestid", "node_id", "node_receive_time", "node_operate_time"
).withColumn(
    "date_range",
    F.expr("sequence(to_date(node_receive_time), to_date(node_operate_time), interval 1 day)")
).withColumn("calc_date", F.explode("date_range")).drop("date_range")

df_time_calc = df_time_calc.withColumn(
    "is_weekday",
    F.dayofweek(F.col("calc_date")).between(2, 6)
).join(
    holiday_df.withColumnRenamed("holiday_date", "calc_date_holiday"),
    F.col("calc_date") == F.col("calc_date_holiday"),
    "left"
).withColumn(
    "is_holiday",
    F.when(F.col("calc_date_holiday").isNotNull(), True).otherwise(False)
).drop("calc_date_holiday").join(
    work_weekend_df.withColumnRenamed("work_weekend_date", "calc_date_work_weekend"),
    F.col("calc_date") == F.col("calc_date_work_weekend"),
    "left"
).withColumn(
    "is_work_weekend",
    F.when(F.col("calc_date_work_weekend").isNotNull(), True).otherwise(False)
).drop("calc_date_work_weekend").withColumn(
    "is_workday",
    F.when(F.col("is_holiday"), False)
     .when(F.col("is_work_weekend"), True)
     .when(F.col("is_weekday"), True)
     .otherwise(False)
).withColumn(
    "day_start_ts",
    F.unix_timestamp(F.to_timestamp(F.col("calc_date")))
).withColumn(
    "day_end_ts",
    F.unix_timestamp(F.concat(F.col("calc_date"), F.lit(" 23:59:59")))
).withColumn(
    "receive_ts",
    F.unix_timestamp(F.col("node_receive_time"))
).withColumn(
    "operate_ts",
    F.unix_timestamp(F.col("node_operate_time"))
).withColumn(
    "daily_receive_ts",
    F.greatest(F.col("receive_ts"), F.col("day_start_ts"))
).withColumn(
    "daily_operate_ts",
    F.least(F.col("operate_ts"), F.col("day_end_ts"))
).withColumn(
    "daily_work_seconds",
    F.when(
        F.col("is_workday"),
        F.greatest(F.lit(0), F.col("daily_operate_ts") - F.col("daily_receive_ts"))
    ).otherwise(F.lit(0))
)

df_node_worktime = df_time_calc.groupBy("requestid", "node_id").agg(
    F.round(F.sum("daily_work_seconds") / 3600, 2).alias("node_workday_cost_hours")
)

# ===================== 关联工作日耗时与最终字段映射 =====================
df_main_with_worktime = df_main \
    .join(df_node_worktime, on=["requestid", "node_id"], how="left") \
    .fillna({"node_cost_hours": 0.0, "node_workday_cost_hours": 0.0})

df_final = df_main_with_worktime.select(
    F.col("requestid").cast(StringType()).alias("request_id"),
    F.col("node_id").cast(StringType()).alias("node_id"),
    F.col("node_code").cast(StringType()).alias("node_code"),
    F.coalesce(F.col("process_name"), F.lit("无流程名称")).alias("process_name"),
    F.col("workflow_name").alias("workflow_name"),
    F.col("process_create_date").alias("process_create_date"),
    F.col("process_status").alias("process_status"),
    F.col("is_return").alias("is_return"),
    F.col("node_remark_info").alias("node_remark_info"),
    F.coalesce(F.col("request_mark"), F.lit("")).alias("request_mark"),
    F.col("requester_dept_full_path").alias("requester_dept_full_path"),
    F.col("requester_dept_name").alias("requester_dept_name"),
    F.col("workflow_id").cast(StringType()).alias("workflow_id"),
    F.col("node_show_order").alias("node_show_order"),
    F.col("node_name").alias("node_name"),
    F.col("node_receive_time").alias("node_receive_time"),
    F.col("node_operate_time").alias("node_operate_time"),
    F.col("node_cost_hours").cast(DoubleType()).alias("node_cost_hours"),
    F.least(
        F.col("node_workday_cost_hours").cast(DoubleType()),
        F.col("node_cost_hours").cast(DoubleType())
    ).alias("node_workday_cost_hours"),
    F.col("approver_user_id").cast(StringType()).alias("approver_user_id"),
    F.coalesce(F.col("approver_name"), F.lit("未知")).alias("approver_name"),
    F.coalesce(F.col("approver_job_level"), F.lit("未知")).alias("approver_job_level"),
    F.coalesce(F.col("approver_job_title"), F.lit("未知")).alias("approver_job_title"),
    F.coalesce(F.col("dept_name"), F.lit("未知")).alias("approver_dept_name"),
    F.coalesce(F.col("dept_full_path"), F.lit("")).alias("approver_dept_full_path"),
    F.coalesce(F.col("sub_company_name"), F.lit("未知")).alias("approver_sub_company_name"),
    F.coalesce(
        F.when(F.instr(F.col("dept_full_path"), "数字化服务中心") > 0, F.lit("IT系统"))
         .when(F.instr(F.col("dept_full_path"), "销售") > 0, F.lit("销售系统"))
         .when(F.instr(F.col("dept_full_path"), "财会") > 0, F.lit("财务系统"))
         .when(F.instr(F.col("dept_full_path"), "生产") > 0, F.lit("生产系统"))
         .when(F.instr(F.col("dept_full_path"), "农务") > 0, F.lit("农务系统"))
         .when(F.instr(F.col("dept_full_path"), "采购") > 0, F.lit("采购系统"))
         .when(F.instr(F.col("dept_full_path"), "审计") > 0, F.lit("内审系统"))
         .when(F.instr(F.col("dept_full_path"), "人力") > 0, F.lit("人事系统"))
         .when(F.instr(F.col("dept_full_path"), "公共关系") > 0, F.lit("总工法系统"))
         .when(F.instr(F.col("dept_full_path"), "质量") > 0, F.lit("质量系统"))
         .otherwise(F.lit("其他")),
        F.lit("其他")
    ).alias("node_user_affiliated_system"),
    F.current_timestamp().alias("dw_update_time"),
    F.lit("OA").alias("source_flg")
)

# ===================== 写入Hive =====================
try:
    # 统计并打印总记录数和唯一 request_id 数
    df_final = df_final.filter(F.col("workflow_name") != "系统提醒工作流")
    total_count = df_final.count()
    distinct_request_ids = df_final.select("request_id").distinct().count()
    print(f"📊 统计结果: 总节点记录数 = {total_count}, 唯一 request_id 数 = {distinct_request_ids}")

    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print(f"✅ 已删除历史表: {target_table}")

    df_final.write.mode("overwrite").format("hive").option("encoding", "UTF-8").saveAsTable(target_table)
    spark.sql(f"""
    ALTER TABLE {target_table} SET TBLPROPERTIES (
        'comment' = 'OA流程效率分析节点明细表（含节点人所属系统、工作日审批时间、提单人部门、request_mark、节点备注）'
    );
    """)
    print(f"✅✅✅ 节点表生成成功！")
    print(f"  - node_cost_hours: 节点总耗时(小时)")
    print(f"  - node_workday_cost_hours: 工作日耗时(小时，剔除周末及节假日，含调休)")
    print(f"  - node_user_affiliated_system: 节点人所属系统")
    print(f"  - requester_dept_full_path: 提单人部门全路径")
    print(f"  - request_mark: 流程标记")
    print(f"  - is_return: 是否退回（精确到节点实例）")
    print(f"  - node_remark_info: 节点备注（保留原始内容，仅去HTML）")
    print(f"✅ 结果表: {target_table}")
except Exception as e:
    print(f"❌ 写入表失败: {e}")
    raise e
finally:
    spark.stop()

/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


📊 统计结果: 总节点记录数 = 4635165, 唯一 request_id 数 = 234216
✅ 已删除历史表: dwd.dwd_oa_workflow_efficiency_nodes_f_1h
✅✅✅ 节点表生成成功！
  - node_cost_hours: 节点总耗时(小时)
  - node_workday_cost_hours: 工作日耗时(小时，剔除周末及节假日，含调休)
  - node_user_affiliated_system: 节点人所属系统
  - requester_dept_full_path: 提单人部门全路径
  - request_mark: 流程标记
  - is_return: 是否退回（精确到节点实例）
  - node_remark_info: 节点备注（保留原始内容，仅去HTML）
✅ 结果表: dwd.dwd_oa_workflow_efficiency_nodes_f_1h
